In [0]:
%sql
CREATE SCHEMA IF NOT EXISTS adb_caresync.gold;

CREATE OR REPLACE TABLE adb_caresync.gold.patient_360 AS
WITH lab_results_agg AS (
  SELECT
    patient_id,
    CAST(COUNT(lab_result_id) AS INT) AS total_tests,
    CAST(COUNT(*) FILTER (WHERE result_status = 'Abnormal') AS INT) AS abnormal_test_count,
    CAST(COUNT(*) FILTER (WHERE result_status = 'Critical') AS INT) AS critical_test_count,
    MAX(test_date) AS last_test_date
  FROM adb_caresync.silver.lab_results
  GROUP BY patient_id
  ORDER BY patient_id
)

SELECT
  p.patient_id,
  p.first_name,
  p.last_name,
  p.date_of_birth,
  DATEDIFF(current_date(), p.date_of_birth) AS age,
  p.gender,
  p.city,
  p.state,
  p.registered_hospital_id AS hospital_id,
  h.hospital_name,
  i.provider_name AS insurance_provider_name,
  l.total_tests,
  l.abnormal_test_count,
  l.critical_test_count,
  l.last_test_date
FROM adb_caresync.silver.patients AS p
LEFT JOIN adb_caresync.silver.hospitals AS h
  ON p.registered_hospital_id = h.hospital_id
LEFT JOIN adb_caresync.silver.insurance_providers AS i
  ON p.insurance_provider_id = i.insurance_provider_id
LEFT JOIN lab_results_agg AS l
  ON p.patient_id = l.patient_id
WHERE l.last_test_date IS NOT NULL
ORDER BY p.patient_id

In [0]:
updated_df = spark.table("adb_caresync.gold.patient_360")
record_count = updated_df.count()
dbutils.notebook.exit(str(record_count))